# Linear Recurrent Neural Network

This notebook implements a simple 1D recurrent rate model to study how activity propagates through a network.

## Idea

We model neurons connected with a Mexican-hat kernel:
- local excitation  
- global inhibition  

We then apply a single-neuron perturbation and measure the resulting influence.

In [ ]:
import numpy as np
from scipy.linalg import circulant
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cdist
from scipy import signal
from pandas import Series
from random import gauss
import ipywidgets as widgets
from ipywidgets import interact, fixed
from src.LinearNeuronModel import compute_firerate, convert_matrix, gaussian, recurrent_connections, autoval_distr2, F
from src.LinearNeuronModel import perturb_M, compute_influence_matrix, compute_variance_across_rows, analyze_all_symmetric
from src.LinearNeuronModel import plot_mean_and_variance_MG, influence_stats

In [ ]:
# Configuration
N = 200
a_e = 2.0
a_i = 1.3
sigma_e = 4.0
r = 2

In [ ]:
# HOMOGENOUS network
M = convert_matrix(recurrent_connections(N, a_e, a_i, sigma_e, r))
M = (0.9 / np.max(np.real(eigvals))) * M

half = N // 2
x    = np.arange(-half, half)
row  = np.roll(M[0], half)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, row, color='steelblue', linewidth=2)
ax.set_xlim(-half, half - 1)
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xlabel('Distance from neuron $i$', fontsize=12)
ax.set_ylabel('Synaptic weight', fontsize=12)

# first and last tick labels bigger
for spine in ax.spines.values():
    spine.set_linewidth(2)

xticks = ax.get_xticks()
ax.set_xticks(xticks)
labels = ax.get_xticklabels()
for idx, label in enumerate(labels):
    if idx == 0 or idx == len(labels) - 1:
        label.set_fontsize(13)
        label.set_fontweight('bold')
    else:
        label.set_fontsize(10)
ax.set_xticklabels(labels)

#plt.suptitle('Connectivity of row M[0]', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("mexican_hat_hom_conn.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
### HETEROGENOUS network
mean_connectivity = np.mean(M)
delta = 10 * mean_connectivity

M_i = M.copy()

# pairs = [(10, 50), (33, 127), (89, 45)]
pairs = [(0, 50), (0, -30), (0, 5)]

for (i, j) in pairs:
    M_i[i, j] += delta
    M_i[j, i] += delta 

fig, axes = plt.subplots(2, 1, figsize=(7, 7), sharex=True)

row_hom = np.roll(M[0], half)
axes[0].plot(x, row_hom, color='steelblue', linewidth=2, label=f'Row {0}')
axes[0].axhline(0, color='gray', linewidth=0.8)
axes[0].set_ylabel('Synaptic weight', fontsize=12)
axes[0].set_title('Homogeneous Kernel - Row 0', fontsize=13, fontweight='bold')
axes[0].text(-0.08, 1.02, '(A)', transform=axes[0].transAxes, fontsize=14, fontweight='bold')
axes[0].set_xlim(-half, half)
for spine in axes[0].spines.values():
    spine.set_linewidth(2)

#colors = ['#C0392B', '#27AE60', '#E67E22']
colors = ['#C0392B']
#for color, (i, j) in zip(colors, pairs):
row_pert = np.roll(M_i[i], half)
axes[1].plot(x, row_pert, color=color, linewidth=2, linestyle='--')
axes[1].axhline(0, color='gray', linewidth=0.8)
axes[1].set_xlabel('Distance from neuron $i$', fontsize=12)
axes[1].set_ylabel('Synaptic weight', fontsize=12)
for spine in ax.spines.values():
    spine.set_linewidth(2)

xticks = ax.get_xticks()
ax.set_xticks(xticks)
labels = ax.get_xticklabels()
for idx, label in enumerate(labels):
    if idx == 0 or idx == len(labels) - 1:
        label.set_fontsize(13)
        label.set_fontweight('bold')
    else:
        label.set_fontsize(10)
ax.set_xticklabels(labels)
axes[1].set_title('Perturbed Kernel - Row 0', fontsize=13, fontweight='bold')
axes[1].text(-0.08, 1.02, '(B)', transform=axes[1].transAxes, fontsize=14, fontweight='bold')
axes[1].set_xlim(-half, half)
#axes[1].legend(fontsize=9)
for spine in axes[1].spines.values():
    spine.set_linewidth(2)

plt.tight_layout(pad=3.0)
plt.savefig("conn_pert_vs_hom.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
eigvals = np.linalg.eigvals(M)
print("Max real eigenvalue:", np.max(np.real(eigvals)))

### INFLUENCE
G0 = compute_influence_matrix(M, N)
G_i = compute_influence_matrix(M_i, N)

fig, axes = plt.subplots(2, 1, figsize=(7, 7), sharex=True)

row_hom = np.roll(G0[0], half)
axes[0].plot(x, row_hom, color='steelblue', linewidth=2, label=f'Row {0}')
axes[0].axhline(0, color='gray', linewidth=0.8)
axes[0].set_ylabel('Synaptic weight', fontsize=12)
axes[0].set_title('Homogeneous Influence - Row 0', fontsize=13, fontweight='bold')
axes[0].text(-0.08, 1.02, '(A)', transform=axes[0].transAxes, fontsize=14, fontweight='bold')
axes[0].set_xlim(-half, half)
for spine in axes[0].spines.values():
    spine.set_linewidth(2)

#colors = ['#C0392B', '#27AE60', '#E67E22']
colors = ['#C0392B']
#for color, (i, j) in zip(colors, pairs):
row_pert = np.roll(G_i[i], half)
axes[1].plot(x, row_pert, color=color, linewidth=2, linestyle='--')
axes[1].axhline(0, color='gray', linewidth=0.8)
axes[1].set_xlabel('Distance from neuron $i$', fontsize=12)
axes[1].set_ylabel('Synaptic weight', fontsize=12)
axes[1].set_title('Perturbed Influence - Row 0', fontsize=13, fontweight='bold')
axes[1].text(-0.08, 1.02, '(B)', transform=axes[1].transAxes, fontsize=14, fontweight='bold')
axes[1].set_xlim(-half, half)
#axes[1].legend(fontsize=9)
for spine in axes[1].spines.values():
    spine.set_linewidth(2)

plt.tight_layout(pad=3.0)
plt.savefig("infl_pert_vs_hom.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
### Variance 
def compute_variance_across_rows(X):
    return np.var(X, axis=1)  

var_i0   = compute_variance_across_rows(G0)
var_c0   = compute_variance_across_rows(M)
var_iper = compute_variance_across_rows(G_i)
var_cper = compute_variance_across_rows(M_i)

neurons = np.arange(len(var_c0))

fig, axes = plt.subplots(2, 1, figsize=(7, 7), sharex=True)

axes[0].plot(neurons, var_c0,   color='steelblue', linewidth=2, label='Homogeneous')
axes[0].plot(neurons, var_cper, color='#C0392B',   linewidth=2, label='Perturbed', linestyle='--')
axes[0].set_ylabel('Variance', fontsize=12)
axes[0].set_title('Connectivity Variance', fontsize=13, fontweight='bold')
axes[0].text(-0.08, 1.02, '(A)', transform=axes[0].transAxes, fontsize=14, fontweight='bold')
axes[0].set_xlim(-half, half)
axes[0].set_xlim(0,N)
axes[0].legend(fontsize=9)
for spine in axes[0].spines.values():
    spine.set_linewidth(2)

axes[1].plot(neurons, var_i0,   color='steelblue', linewidth=2, label='Homogeneous')
axes[1].plot(neurons, var_iper, color='#C0392B',   linewidth=2, label='Perturbed', linestyle='--')
axes[1].set_xlabel('Neuron $i$', fontsize=12)
axes[1].set_ylabel('Variance', fontsize=12)
axes[1].set_title('Influence Variance', fontsize=13, fontweight='bold')
axes[1].text(-0.08, 1.02, '(B)', transform=axes[1].transAxes, fontsize=14, fontweight='bold')
axes[1].set_xlim(0,N)
axes[1].legend(fontsize=9)
for spine in axes[1].spines.values():
    spine.set_linewidth(2)

plt.tight_layout()
plt.savefig("variance.png", dpi=180, bbox_inches="tight")
plt.show()